In [183]:
import pandas as pd
import sqlite3

In [184]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

In [185]:
conn.execute("DROP TABLE IF EXISTS datamart;")
print("Существующая таблица datamart удалена (если она была)")

Существующая таблица datamart удалена (если она была)


In [186]:
create_datamart_query = """
CREATE TABLE datamart AS
SELECT 
    CASE WHEN c.uid LIKE 'user_%' THEN c.uid ELSE NULL END AS uid,
    CASE WHEN c.uid LIKE 'user_%' THEN c.labname ELSE NULL END AS labname,
    CASE WHEN c.uid LIKE 'user_%' THEN MIN(c.timestamp) ELSE NULL END AS first_commit_ts,
    CASE WHEN c.uid LIKE 'user_%' THEN MIN(p.datetime) ELSE NULL END AS first_view_ts
FROM checker c
LEFT JOIN pageviews p ON c.uid = p.uid
WHERE c.status = 'ready'
    AND c.numTrials = 1
    AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
GROUP BY c.uid, c.labname
ORDER BY c.uid, c.labname;
"""

conn.execute(create_datamart_query)

In [187]:
check_table_query = "SELECT name FROM sqlite_master WHERE type='table' AND name='datamart';"
table_exists = pd.io.sql.read_sql(check_table_query, conn)
print("Таблица datamart существует:", not table_exists.empty)

Таблица datamart существует: True


In [188]:
datamart = pd.io.sql.read_sql("SELECT * FROM datamart;", conn)

datamart['first_commit_ts'] = pd.to_datetime(datamart['first_commit_ts'])
datamart['first_view_ts'] = pd.to_datetime(datamart['first_view_ts'])

print("Информация о DataFrame datamart:")
print(datamart.info())

Информация о DataFrame datamart:
<class 'pandas.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    str           
 1   labname          140 non-null    str           
 2   first_commit_ts  140 non-null    datetime64[us]
 3   first_view_ts    59 non-null     datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 4.7 KB
None


In [189]:
test = datamart[datamart['first_view_ts'].notna() & datamart['uid'].notna()].copy()
print(f"Размер test: {len(test)} записей")
print("Первые 5 строк test:")
print(test.head())

Размер test: 59 записей
Первые 5 строк test:
       uid  labname            first_commit_ts              first_view_ts
6   user_1   laba04 2020-04-26 17:06:18.462708 2020-04-26 21:53:59.624136
7   user_1  laba04s 2020-04-26 17:12:11.843671 2020-04-26 21:53:59.624136
8   user_1   laba05 2020-05-02 19:15:18.540185 2020-04-26 21:53:59.624136
9   user_1   laba06 2020-05-17 16:26:35.268534 2020-04-26 21:53:59.624136
10  user_1  laba06s 2020-05-20 12:23:37.289724 2020-04-26 21:53:59.624136


In [190]:
control = datamart[~datamart.index.isin(test.index)].copy()
print(f"Размер control: {len(control)} записей")
print("Первые 5 строк control:")
print(control.head())

Размер control: 87 записей
Первые 5 строк control:
   uid labname first_commit_ts first_view_ts
0  NaN     NaN             NaT           NaT
1  NaN     NaN             NaT           NaT
2  NaN     NaN             NaT           NaT
3  NaN     NaN             NaT           NaT
4  NaN     NaN             NaT           NaT


In [191]:
mean_view_ts = test['first_view_ts'].mean()
print(f"Среднее значение first_view_ts в test: {mean_view_ts}")

control.loc[control['uid'].notna(), 'first_view_ts'] = control.loc[control['uid'].notna(), 'first_view_ts'].fillna(mean_view_ts)

print("\nControl после замены пропусков:")
print(control.head())
print(f"\nКоличество пропусков в control после замены: {control['first_view_ts'].isna().sum()}")

Среднее значение first_view_ts в test: 2020-04-27 00:40:05.761783

Control после замены пропусков:
   uid labname first_commit_ts first_view_ts
0  NaN     NaN             NaT           NaT
1  NaN     NaN             NaT           NaT
2  NaN     NaN             NaT           NaT
3  NaN     NaN             NaT           NaT
4  NaN     NaN             NaT           NaT

Количество пропусков в control после замены: 6


In [192]:
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)
print("Таблицы 'test' и 'control' успешно сохранены в базу данных")

Таблицы 'test' и 'control' успешно сохранены в базу данных


In [193]:
test.info()

<class 'pandas.DataFrame'>
Index: 59 entries, 6 to 120
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     str           
 1   labname          59 non-null     str           
 2   first_commit_ts  59 non-null     datetime64[us]
 3   first_view_ts    59 non-null     datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 2.3 KB


In [194]:
control.info()

<class 'pandas.DataFrame'>
Index: 87 entries, 0 to 145
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     str           
 1   labname          81 non-null     str           
 2   first_commit_ts  81 non-null     datetime64[us]
 3   first_view_ts    81 non-null     datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 3.4 KB


In [195]:
test = pd.io.sql.read_sql('SELECT * FROM test', conn)
control = pd.io.sql.read_sql('SELECT * FROM control', conn)

In [196]:
conn.close()